In [5]:
#!/usr/bin/env python
import copy
import coffea
import numpy as np
import awkward as ak
import json
import hist
import yaml

# from mt2 import mt2

from coffea import processor
from coffea.analysis_tools import PackedSelection
from coffea.nanoevents.methods import vector
from coffea.lumi_tools import LumiMask

# silence warnings due to using NanoGEN instead of full NanoAOD
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

from topcoffea.modules.paths import topcoffea_path
from topcoffea.modules.histEFT import HistEFT
import topcoffea.modules.eft_helper as efth
import topcoffea.modules.corrections as tc_cor
import topcoffea.modules.event_selection as tc_es

from ttbarEFT.modules.paths import ttbarEFT_path
from ttbarEFT.modules.analysis_tools import make_mt2, get_lumi
import ttbarEFT.modules.object_selection as tt_os
import ttbarEFT.modules.event_selection as tt_es
import ttbarEFT.modules.corrections as tt_cor 
from ttbarEFT.modules.processor_tools import calc_eft_weights
from ttbarEFT.modules.processor_tools import get_syst_lists

from ttbarEFT.modules import plotting_tools_histEFT as plt_tools

from topcoffea.modules.get_param_from_jsons import GetParam

get_tc_param = GetParam(topcoffea_path("params/params.json"))
get_tt_param = GetParam(ttbarEFT_path("params/params.json"))

NanoAODSchema.warn_missing_crossrefs = False
np.seterr(divide='ignore', invalid='ignore', over='ignore')

import pickle, gzip
import matplotlib.pyplot as plt
from cycler import cycler
import mplhep as hep

import hist
from hist import Hist

from topcoffea.modules.histEFT import HistEFT

In [6]:
def load_pickle(fname):
    return pickle.load(gzip.open(fname))

In [18]:
hist_dict = load_pickle("SR_ALL_260510.pkl.gz")

In [33]:
set(MC_procs) == set(TT_procs+tW_procs+DY_procs+Others_procs)

True

In [ ]:
for ch in hist_dict.keys():
    var = 'mllbb'
    hists = hist_dict[ch][var].as_hist({})
    channels = list(hists.axes['channel'])
    all_procs = list(hists.axes['process'])
    MC_procs = [x for x in all_procs if ('data' not in x)]
    
    TT_procs = [x for x in MC_procs if "TT01j2l" in x]
    tW_procs = [x for x in MC_procs if "TW" in x]
    DY_procs = [x for x in MC_procs if "DY" in x]
    TTGJets_procs = [x for x in MC_procs if "TTG" in x]
    ttW_procs = [x for x in MC_procs if "ttW" in x]
    ttZ_procs = [x for x in MC_procs if "ttZ" in x]
    Others_procs = [x for x in MC_procs if ((x not in TT_procs) and (x not in tW_procs) and (x not in DY_procs))]
    
    for chan in channels: 
        print(f"channel: {chan}")        
        TT_vals = hists[{'systematic':'nominal', 'process':TT_procs, 'channel':chan}][{'process':sum}].values()
        tW_Vals = hists[{'systematic':'nominal', 'process':tW_procs, 'channel':chan}][{'process':sum}].values()
        DY_vals = hists[{'systematic':'nominal', 'process':DY_procs, 'channel':chan}][{'process':sum}].values()
        TTG_vals = hists[{'systematic':'nominal', 'process':TTGJets_procs, 'channel':chan}][{'process':sum}].values()
        ttW_vals = hists[{'systematic':'nominal', 'process':ttW_procs, 'channel':chan}][{'process':sum}].values()
        ttZ_vals = hists[{'systematic':'nominal', 'process':ttZ_procs, 'channel':chan}][{'process':sum}].values()
        Others_vals = hists[{'systematic':'nominal', 'process':Others_procs, 'channel':chan}][{'process':sum}].values()

        total = TT_vals[TT_vals >= 0].sum() + tW_Vals[tW_Vals >= 0].sum() + DY_vals[DY_vals >= 0].sum() + Others_vals[Others_vals >= 0].sum()
        print(f"total events: {total}")
        print(f"\t ttbar events: {TT_vals[TT_vals >= 0].sum()}")
        print(f"\t tW events: {tW_Vals[tW_Vals >= 0].sum()}")
        print(f"\t DY events: {DY_vals[DY_vals >= 0].sum()}")
        print(f"\t Others events: {Others_vals[Others_vals >= 0].sum()}")
        print(f"\t TTG events: {TTG_vals[TTG_vals >= 0].sum()}")
        print(f"\t ttW events: {ttW_vals[ttW_vals >= 0].sum()}")
        print(f"\t ttZ events: {ttZ_vals[ttZ_vals >= 0].sum()}")
        # print(f"total events: {ak.sum(nom_hist.values())}")
        # print(f"\t ttbar events: {ak.sum(hists[{'systematic':'nominal', 'process':TT_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t tW events: {ak.sum(hists[{'systematic':'nominal', 'process':tW_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t DY events: {ak.sum(hists[{'systematic':'nominal', 'process':DY_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t Others events: {ak.sum(hists[{'systematic':'nominal', 'process':Others_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t TTG events: {ak.sum(hists[{'systematic':'nominal', 'process':TTGJets_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t TTZ events: {ak.sum(hists[{'systematic':'nominal', 'process':ttZ_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t TTW events: {ak.sum(hists[{'systematic':'nominal', 'process':ttW_procs, 'channel':chan}][{'process':sum}].values())}")

channel: ee_2b_2j
total events: 12942.334428787653
	 ttbar events: 12324.265625051978
	 tW events: 477.29305390471524
	 DY events: 65.1207449960475
	 Others events: 75.65500483491357
	 TTG events: 58.97779191112984
	 ttW events: 6.778479650192336
	 ttZ events: 7.855692287504297
channel: ee_2b_3j
total events: 8655.43148298275
	 ttbar events: 8238.100514920618
	 tW events: 306.6981858156222
	 DY events: 25.1888325580252
	 Others events: 85.44394968848619
	 TTG events: 62.36285237880831
	 ttW events: 10.82903620931186
	 ttZ events: 10.097949402738088
channel: ee_2b_4j
total events: 5936.828830058268
	 ttbar events: 5647.855205379178
	 tW events: 167.28183575419303
	 DY events: 22.83074919180881
	 Others events: 98.86103973308894
	 TTG events: 64.37534489566383
	 ttW events: 15.846067081896262
	 ttZ events: 16.471849162353685
channel: mm_2b_2j
total events: 10920.073480133586
	 ttbar events: 10350.596605804281
	 tW events: 433.89606370521426
	 DY events: 56.91436808371588
	 Others events: